In [3]:
import asyncio
from codecs import StreamReader
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent

from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from dotenv import load_dotenv
from autogen_agentchat.ui import Console
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

python-dotenv could not parse statement starting at line 5
python-dotenv could not parse statement starting at line 6


In [4]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat


planning_agent = AssistantAgent(
    name="PlanningAgent",
    description="An agent for planning tasks, this agent should be the first to engage when given a new task.",
    model_client=model_client,
    system_message="""
    You are a planning agent.
    Your job is to break down complex tasks into smaller, manageable subtasks.
    Your team members are:
        WebSearchAgent: Searches for information
        DataAnalystAgent: Performs calculations

    You only plan and delegate tasks - you do not execute them yourself.

    When assigning tasks, use this format:
    1. <agent> : <task>

    After all tasks are complete, summarize the findings and end with "TERMINATE".
    """,
)

In [5]:
from dotenv import load_dotenv

from langchain_community.utilities import GoogleSerperAPIWrapper

from autogen_ext.tools.http import HttpTool


os.environ['SERPER_API_KEY']='bead05022450578faa7498f4c90d85e534c372e0'


search_tool_wrapper = GoogleSerperAPIWrapper(type='search')

def search_web(query:str) ->str:
    """Search the web for the given query and return the results."""
    try:
        results = search_tool_wrapper.run(query)
        return results
    except Exception as e:
        print(f"Error occurred while searching the web: {e}")
        return "No results found."

In [6]:
def search_web_tool(query:str)-> str:
    # Simulate a web search
    if "2006-2007" in query:
        return """Here are the total points scored by Miami Heat players in the 2006-2007 season:
        Udonis Haslem: 844 points
        Dwayne Wade: 1397 points
        James Posey: 550 points
        ...
        """
    elif "2007-2008" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2007-2008 is 214."
    elif "2008-2009" in query:
        return "The number of total rebounds for Dwayne Wade in the Miami Heat season 2008-2009 is 398."
    return "No data found."

In [7]:
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)


In [8]:
web_search_agent = AssistantAgent(
    name = 'WebSearchAgent',
    description= 'An agent for searching the web for information.',
    model_client=model_client,
    tools = [search_web_tool],
    reflect_on_tool_use=False,
    system_message='''
        You are a web search agent.
        Your only tool is search_web - use it to find the information you need.

        You make only one search call at a time.
        
        Once you have the results, you never do calculations or data analysis on them.
    ''',
)

In [9]:
def percentage_change_tool(start:float, end:float) -> float:
    # Calculate percentage change
    if start == 0:
        return 0
    return ((end - start) / start) * 100

In [10]:
data_analyst_agent = AssistantAgent(
    name = 'DataAnalystAgent',
    description= 'An agent for performing calculations and data analysis.',
    model_client=model_client,
    tools= [percentage_change_tool],
    system_message='''
        You are a data analyst agent.
        Given the tasks you have been assigned, you should analyze the data and provide results using the tools provided (percentage_change_tool).

        If you have not seen the data, ask for it.

    ''',
)

Termination Condition


In [11]:
from autogen_agentchat.conditions import TextMentionTermination,MaxMessageTermination

text_mention_termination = TextMentionTermination('TERMINATE')
max_message_termination = MaxMessageTermination(max_messages=20)
combined_termination = text_mention_termination | max_message_termination

In [12]:
selector_prompt = '''
Select an agent to perform the task.

{roles}

current conversation history :
{history}

Read the above conversation, then select an agent from {participants} to perform the next task.
Make sure that the planning agent has assigned task before other agents start working.
Only select one agent.
'''

In [13]:
planning_agent.description


'An agent for planning tasks, this agent should be the first to engage when given a new task.'

In [14]:
selector_team = SelectorGroupChat(
    participants=[planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=combined_termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True)

In [15]:
task = "Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"


In [16]:
from autogen_agentchat.ui import Console

await Console(selector_team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To find the answer to your question, we need to take the following steps:

1. WebSearchAgent: Search for the Miami Heat player with the highest points scored in the 2006-2007 NBA season.
2. WebSearchAgent: Gather the total rebounds statistics for this player for the 2007-2008 NBA season.
3. WebSearchAgent: Gather the total rebounds statistics for the same player for the 2008-2009 NBA season.
4. DataAnalystAgent: Calculate the percentage change in total rebounds between the 2007-2008 and 2008-2009 seasons for this player.

Let's proceed with the tasks: 

1. WebSearchAgent: Search for the Miami Heat player with the highest points scored in the 2006-2007 NBA season.
---------- ToolCallRequestEvent (WebSearchAgent) --------

TaskResult(messages=[TextMessage(id='2d70aab6-0562-4c3f-b243-896cecc7df3c', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 19, 8, 4, 9, 240135, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='b905598a-3bdf-430a-b76f-42300ae6297f', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=161, completion_tokens=164), metadata={}, created_at=datetime.datetime(2025, 9, 19, 8, 4, 15, 261863, tzinfo=datetime.timezone.utc), content="To find the answer to your question, we need to take the following steps:\n\n1. WebSearchAgent: Search for the Miami Heat player with the highest points scored in the 2006-2007 NBA season.\n2. WebSearchAgent: Gather the total rebounds statistics for this player for the 2007-2008 NBA season.\n3. WebSearchAgent: Ga

In [18]:
# With real web search


from autogen_agentchat.ui import Console

await Console(selector_team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To answer your question, we need to conduct the following steps:

1. WebSearchAgent: Identify which Miami Heat player scored the highest points during the 2006-2007 NBA season.
2. WebSearchAgent: Find the total rebounds for this player during the 2007-2008 NBA season.
3. WebSearchAgent: Find the total rebounds for the same player during the 2008-2009 NBA season.
4. DataAnalystAgent: Calculate the percentage change in total rebounds between the 2007-2008 and 2008-2009 seasons for the player.

Now, let's assign these tasks:

1. WebSearchAgent: Identify the Miami Heat player with the highest points scored in the 2006-2007 NBA season.
---------- ThoughtEvent (WebSearchAgent) ----------
The Miami Heat player with the highest

TaskResult(messages=[TextMessage(id='ce6963a8-39df-4d68-8a21-3505c3a8df4a', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 19, 8, 5, 55, 943576, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='0d6ccfbf-71ab-4331-ac54-74e4467289dd', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=733, completion_tokens=156), metadata={}, created_at=datetime.datetime(2025, 9, 19, 8, 5, 59, 213223, tzinfo=datetime.timezone.utc), content="To answer your question, we need to conduct the following steps:\n\n1. WebSearchAgent: Identify which Miami Heat player scored the highest points during the 2006-2007 NBA season.\n2. WebSearchAgent: Find the total rebounds for this player during the 2007-2008 NBA season.\n3. WebSearchAgent: Find the total reboun

In [19]:
state = await selector_team.save_state()


In [20]:
state

{'type': 'TeamState',
 'version': '1.0.0',
 'agent_states': {'PlanningAgent': {'type': 'ChatAgentContainerState',
   'version': '1.0.0',
   'agent_state': {'type': 'AssistantAgentState',
    'version': '1.0.0',
    'llm_context': {'messages': [{'content': 'Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?',
       'source': 'user',
       'type': 'UserMessage'},
      {'content': "To find the answer to your question, we need to take the following steps:\n\n1. WebSearchAgent: Search for the Miami Heat player with the highest points scored in the 2006-2007 NBA season.\n2. WebSearchAgent: Gather the total rebounds statistics for this player for the 2007-2008 NBA season.\n3. WebSearchAgent: Gather the total rebounds statistics for the same player for the 2008-2009 NBA season.\n4. DataAnalystAgent: Calculate the percentage change in total rebounds between the 200

In [21]:
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from typing import Sequence

def my_selector_fun(messages: Sequence[BaseAgentEvent | BaseChatMessage]):

    if messages[-1].source == web_search_agent.name:
        return data_analyst_agent.name
    return None


selector_team = SelectorGroupChat(
    participants=[planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=combined_termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=True,
    selector_func=my_selector_fun)

In [22]:
task = "Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"

from autogen_agentchat.ui import Console

await Console(selector_team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (PlanningAgent) ----------
To answer your question, the steps executed are as follows:

1. **Identify the highest-scoring player for the Miami Heat during the 2006-2007 NBA season**: 
   - **Dwyane Wade** was identified as the player with the highest points, scoring 1,397 points.

2. **Retrieve total rebounds for both seasons**:
   - For the 2007-2008 NBA season, Dwyane Wade had a total of 214 rebounds.
   - For the 2008-2009 NBA season, he had a total of 398 rebounds.

3. **Calculate the percentage change in total rebounds**:
   - The calculated percentage increase in rebounds from the 2007-2008 season to the 2008-2009 season was approximately **85.98%**.

Thus, Dwyane Wade was the Miami Heat's highest scorer in the 2006-2007 season, and he expe

TaskResult(messages=[TextMessage(id='8bdd77b5-bee6-409b-8c4c-993d2fc147d8', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 19, 9, 14, 0, 220593, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='c698e8d0-40fc-4584-b79a-ad4b6cd046c0', source='PlanningAgent', models_usage=RequestUsage(prompt_tokens=1444, completion_tokens=226), metadata={}, created_at=datetime.datetime(2025, 9, 19, 9, 14, 4, 468252, tzinfo=datetime.timezone.utc), content="To answer your question, the steps executed are as follows:\n\n1. **Identify the highest-scoring player for the Miami Heat during the 2006-2007 NBA season**: \n   - **Dwyane Wade** was identified as the player with the highest points, scoring 1,397 points.\n\n2. **Retrieve total rebounds for both seasons**:\n  

In [23]:

await selector_team.reset()


In [25]:
task = "Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?"

from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from typing import Sequence

def my_selector_fun(messages: Sequence[BaseAgentEvent | BaseChatMessage]):

    if messages[-1].source != web_search_agent.name:
        return data_analyst_agent.name
    return None


selector_team = SelectorGroupChat(
    participants=[planning_agent, web_search_agent, data_analyst_agent],
    model_client=model_client,
    termination_condition=combined_termination,
    selector_prompt=selector_prompt,
    allow_repeated_speaker=False,
    selector_func=my_selector_fun)


from autogen_agentchat.ui import Console

await Console(selector_team.run_stream(task=task))

---------- TextMessage (user) ----------
Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?
---------- TextMessage (DataAnalystAgent) ----------
Please provide the data for the Miami Heat player's points for the 2006-2007 season and their total rebounds for the 2007-2008 and 2008-2009 seasons.
---------- TextMessage (DataAnalystAgent) ----------
Please provide the data for the Miami Heat player's points for the 2006-2007 season and their total rebounds for the 2007-2008 and 2008-2009 seasons.
---------- ToolCallRequestEvent (DataAnalystAgent) ----------
[FunctionCall(id='call_Vt5bGhSlMu91k5SE0mz8E3ui', arguments='{"start": 2007, "end": 2008}', name='percentage_change_tool'), FunctionCall(id='call_sr5xdLCCgKwuViTd3VaMl6Ki', arguments='{"start": 2008, "end": 2009}', name='percentage_change_tool')]
---------- ToolCallExecutionEvent (DataAnalystAgent) ----------


TaskResult(messages=[TextMessage(id='7c4c94f1-5ce6-46a5-90db-caf52dedf7f5', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 19, 9, 14, 58, 653074, tzinfo=datetime.timezone.utc), content='Who was the Miami Heat player with the highest point in the 2006-2007 season, and what was the percentage change in his total rebounds between the 2007-2008 and 2008-2009 seasons?', type='TextMessage'), TextMessage(id='5ff946eb-d9a1-4c26-adf2-98bc788941e3', source='DataAnalystAgent', models_usage=RequestUsage(prompt_tokens=150, completion_tokens=41), metadata={}, created_at=datetime.datetime(2025, 9, 19, 9, 15, 1, 426961, tzinfo=datetime.timezone.utc), content="Please provide the data for the Miami Heat player's points for the 2006-2007 season and their total rebounds for the 2007-2008 and 2008-2009 seasons.", type='TextMessage'), TextMessage(id='a4ba4616-2e00-4375-a40f-7f28ba4620f7', source='DataAnalystAgent', models_usage=RequestUsage(prompt_tokens=194, completion